In [31]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/house-prices-advanced-regression-techniques/sample_submission.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/data_description.txt
/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv
/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv


In [32]:
# Install required packages (if not already installed)
!pip install mlflow dagshub -q

# Import libraries
import mlflow
import dagshub
from kaggle_secrets import UserSecretsClient
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

# Load secrets from Kaggle
user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("ML_assn1")   # your DagsHub token
dagshub_username = user_secrets.get_secret("username")  # your DagsHub username

# Set environment variables for authentication
import os
os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# Initialize DagsHub + MLflow
repo_owner = dagshub_username
repo_name = "ml_assn1"

mlflow.set_tracking_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
mlflow.set_registry_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")

# Optional: initialize dagshub (helps with repo linking)
dagshub.init(repo_name=repo_name, repo_owner=repo_owner, mlflow=True)

# Test connection by starting a run
with mlflow.start_run():
    mlflow.log_param("test_param", 123)
    mlflow.log_metric("test_metric", 0.95)

print("✅ Connected to DagsHub + MLflow successfully!")

Initialized MLflow to track repo "lkuch23/ml_assn1"

Repository lkuch23/ml_assn1 initialized!

🏃 View run treasured-lamb-325 at: https://dagshub.com/lkuch23/ml_assn1.mlflow/#/experiments/0/runs/278040b79979469fa832a366959bcbca
🧪 View experiment at: https://dagshub.com/lkuch23/ml_assn1.mlflow/#/experiments/0
✅ Connected to DagsHub + MLflow successfully!


In [33]:
train = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')

Cleaning

In [34]:
y = np.log1p(train["SalePrice"])
train = train.drop("SalePrice", axis=1)

In [35]:
num_cols = train.select_dtypes(include=['int64', 'float64']).columns
cat_cols = train.select_dtypes(include=['object']).columns

In [36]:
train[num_cols] = train[num_cols].fillna(train[num_cols].median())
train[cat_cols] = train[cat_cols].fillna("None")

Feature Engineering

In [37]:
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF'] 
train['TotalBathSF'] = train['FullBath'] + train['BsmtFullBath'] + 0.5 * (train['HalfBath'] + train['BsmtHalfBath'])
train['TotalRooms'] = train['TotRmsAbvGrd'] + train['FullBath'] + train['HalfBath']

Feature Selection

In [38]:
train = pd.get_dummies(train)

In [39]:
X = train.iloc[:len(train)]

Training

In [40]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [41]:
import warnings
with mlflow.start_run() as run:
    model = GradientBoostingRegressor(n_estimators=300,learning_rate=0.05,max_depth=3,random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mlflow.log_param("model", "GradientBoosting")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 3)
    mlflow.log_metric("rmse", rmse)
    final_model = GradientBoostingRegressor(n_estimators=300,learning_rate=0.05,max_depth=3,random_state=42)
    final_model.fit(X, y)
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")
        result = mlflow.sklearn.log_model(
            sk_model              = final_model,
            name                  = "model",
            registered_model_name = "house-prices-gb",
        )

    RUN_ID = run.info.run_id
    print(f"RMSE      : {rmse:.5f}")
    print(f"Run ID    : {RUN_ID}")
    print(f"Version   : {result.registered_model_version}")

2026/04/13 08:56:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'house-prices-gb' already exists. Creating a new version of this model...
2026/04/13 08:56:57 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: house-prices-gb, version 5
Created version '5' of model 'house-prices-gb'.


RMSE      : 0.13589
Run ID    : 513ccf0a5f4042168e745969a9f08591
Version   : 5
🏃 View run smiling-stag-306 at: https://dagshub.com/lkuch23/ml_assn1.mlflow/#/experiments/0/runs/513ccf0a5f4042168e745969a9f08591
🧪 View experiment at: https://dagshub.com/lkuch23/ml_assn1.mlflow/#/experiments/0
